# Part 3 · Waste Description Classification
**Owner: Jeff**

### Setup (VS Code / local) — run first

In [ ]:
import os, zipfile, pathlib, shutil
# Unzip data/realwaste.zip -> data/RealWaste/<category>/ if not already done
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z: z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p),"data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)
IMAGE_DIR = "data/RealWaste"
print("Images present:", os.path.isdir(IMAGE_DIR))

In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

### 3.1 Load & split text data

In [ ]:
from sklearn.model_selection import train_test_split
desc = pd.read_csv("data/waste_descriptions.csv")
class_names = sorted(desc["category"].unique())
def clean_text(t):
    t=str(t).lower(); t=re.sub(r"[^a-z\s]"," ",t); return re.sub(r"\s+"," ",t).strip()
train_txt, test_txt = train_test_split(desc, test_size=0.2, stratify=desc["category"], random_state=42)
print(len(train_txt),"train /",len(test_txt),"test")

### 3.2 TF-IDF + Logistic Regression (Option A)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([("tfidf", TfidfVectorizer(preprocessor=clean_text, ngram_range=(1,2),
                                           min_df=2, max_features=20000, stop_words="english")),
                 ("clf", LogisticRegression(max_iter=1000, C=5.0, class_weight="balanced"))])
pipe.fit(train_txt["description"], train_txt["category"])

### 3.3 Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
pred = pipe.predict(test_txt["description"])
print("Test accuracy:", accuracy_score(test_txt["category"], pred))
print(classification_report(test_txt["category"], pred))
sns.heatmap(confusion_matrix(test_txt["category"],pred,labels=class_names),annot=True,fmt="d",
            xticklabels=class_names,yticklabels=class_names)
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

### 3.4 Classification function (grader interface)

In [ ]:
def classify_waste_description(description):
    return str(pipe.predict([description])[0])
classify_waste_description("greasy pizza box with cheese stuck on it")

### TODO (Jeff)\n- Inspect misclassifications / confusion cases\n- OPTIONAL Option B: fine-tune DistilBERT and compare\n- Justify the architecture choice